In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
np.random.seed(99)
n = 800
df = pd.DataFrame({
      "purchase_value":  np.append(np.random.exponential(scale=120, size=792), [7200, 9500, 11000, 8800, 10200, 12500, 6900, 9100]),
    "delivery_days":   np.append(np.abs(np.random.normal(loc=6, scale=2, size=793)), [44, 58, 63, 50, 47, 55, 7]),
    "review_score":    np.clip(np.append(np.random.normal(loc=4.0, scale=0.6, size=799), [1]), 1, 5).round(1)
})
print(df.shape)
print(df.describe())

(800, 3)
       purchase_value  delivery_days  review_score
count      800.000000     800.000000    800.000000
mean       222.260039       6.474309      3.978375
std        948.594784       4.550191      0.593262
min          0.002088       0.196098      1.000000
25%         36.898998       4.740409      3.600000
50%         90.173587       6.214954      4.000000
75%        181.334721       7.570977      4.400000
max      12500.000000      63.000000      5.000000


Q1 - Profile and choose your detection method
For each column, compare mean vs median from describe() and decide whether to use Z-score or IQR. Write your decision as a comment.

for col in ["purchase_value", "delivery_days", "review_score"]:
    desc = df[col].describe()
    # Print mean, median, and % difference
    # Decide: Z-score or IQR?
    

In [10]:
for col in ["purchase_value", "delivery_days", "review_score"]:
    desc = df[col].describe()

    mean = desc["mean"]
    median = desc["50%"]

    percent_difference = abs(mean - median) / median * 100

    print("\nColumn:", col)
    print("Mean:", mean)
    print("Median:", median)
    print("Difference:", percent_difference, "%")

    if percent_difference > 15:
        print("Decision: Use IQR because the column is skewed.")
    else:
        print("Decision: Use Z-score because the column is roughly symmetric.")


Column: purchase_value
Mean: 222.26003862942449
Median: 90.17358661360794
Difference: 146.4802022146512 %
Decision: Use IQR because the column is skewed.

Column: delivery_days
Mean: 6.474308911770614
Median: 6.214954041956508
Difference: 4.173077838761613 %
Decision: Use Z-score because the column is roughly symmetric.

Column: review_score
Mean: 3.9783749999999998
Median: 4.0
Difference: 0.5406250000000057 %
Decision: Use Z-score because the column is roughly symmetric.


Q2 - Detect outliers in delivery_days using Z-score
delivery_days is roughly symmetric. Apply Z-score detection and print the flagged rows.

z = stats.zscore(df["delivery_days"])
# Create a boolean mask where |z| > 3
# Print count of flagged rows and the flagged values

In [12]:
z = stats.zscore(df["delivery_days"])
outlier_mask_delivery = np.abs(z) > 3
print("Number of outliers:", outlier_mask_delivery.sum())
print("\nFlagged rows:")
print(df[outlier_mask_delivery])

Number of outliers: 6

Flagged rows:
     purchase_value  delivery_days  review_score
793          9500.0           44.0           4.2
794         11000.0           58.0           4.4
795          8800.0           63.0           3.2
796         10200.0           50.0           4.4
797         12500.0           47.0           3.1
798          6900.0           55.0           4.3


Q3 - Detect outliers in purchase_value using IQR
purchase_value is right-skewed. Calculate Q1, Q3, IQR, fences, and flag rows outside the fences.

Q1 = df["purchase_value"].quantile(0.25)
Q3 = df["purchase_value"].quantile(0.75)
# Calculate IQR, lower_fence, upper_fence
# Create outlier mask and print flagged rows

In [13]:
Q1 = df["purchase_value"].quantile(0.25)
Q3 = df["purchase_value"].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
outlier_mask_purchase = (
    (df["purchase_value"] < lower_fence) |
    (df["purchase_value"] > upper_fence)
)
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower fence:", lower_fence)
print("Upper fence:", upper_fence)
print("Number of outliers:", outlier_mask_purchase.sum())
print("\nFlagged rows:")
print(df[outlier_mask_purchase])

Q1: 36.89899833995587
Q3: 181.33472059414868
IQR: 144.4357222541928
Lower fence: -179.7545850413333
Upper fence: 397.9883039754379
Number of outliers: 46

Flagged rows:
     purchase_value  delivery_days  review_score
8        560.395762       7.856667           4.2
16       437.757511       6.648644           4.3
47       414.072407       4.666291           4.0
62       449.842273       1.581624           4.1
104      644.891199       6.551485           3.4
137      406.016583       4.104994           4.0
213      525.037418       4.384138           4.0
223      546.868518       4.252674           3.7
243      401.822466       7.795741           4.3
252      545.441443       7.985127           3.7
253      453.170321       5.525483           4.7
281      884.239258       6.705352           3.5
301      449.582406       6.969827           4.8
310      399.513071       6.626842           4.4
345      514.762691       7.182518           4.5
361      859.463624       8.865819           4.

Q4 - Treat delivery_days outliers: Remove
The operations team confirms: delivery times above 40 days are system logging errors - the order was delivered on time but the status was never updated. Drop those rows.

# Use your mask from Q2 to drop flagged rows
# Print shape before and after
# Print describe() before and after and note what changed in max and mean

In [16]:
df_clean  =  df[~outlier_mask_delivery].copy()
print("Shape before:", df.shape)
print("Shape after:", df_clean.shape)
print("\nBefore:")
print(df["delivery_days"].describe())
print("\nAfter:")
print(df_clean["delivery_days"].describe())

Shape before: (800, 3)
Shape after: (794, 3)

Before:
count    800.000000
mean       6.474309
std        4.550191
min        0.196098
25%        4.740409
50%        6.214954
75%        7.570977
max       63.000000
Name: delivery_days, dtype: float64

After:
count    794.000000
mean       6.123989
std        2.038548
min        0.196098
25%        4.737017
50%        6.194851
75%        7.543245
max       13.534982
Name: delivery_days, dtype: float64


Q5 - Treat purchase_value outliers: Cap/Floor
The business team confirms the high purchase values are real - bulk orders from corporate buyers. Removing them would erase valid transactions. Cap at the IQR upper fence instead.

# Use .clip() with lower_fence and upper_fence from Q3
# Store result in df["purchase_value_capped"]
# Print describe() for both columns side by side
# Verify: max of capped column == upper_fence

In [17]:
df["purchase_value_capped"] = df["purchase_value"].clip(
    lower=lower_fence,
    upper=upper_fence
)
print("Original purchase_value:")
print(df["purchase_value"].describe())
print("\nCapped purchase_value:")
print(df["purchase_value_capped"].describe())
print("\nUpper fence:", upper_fence)
print("Maximum after capping:", df["purchase_value_capped"].max())

Original purchase_value:
count      800.000000
mean       222.260039
std        948.594784
min          0.002088
25%         36.898998
50%         90.173587
75%        181.334721
max      12500.000000
Name: purchase_value, dtype: float64

Capped purchase_value:
count    800.000000
mean     125.117912
std      111.347550
min        0.002088
25%       36.898998
50%       90.173587
75%      181.334721
max      397.988304
Name: purchase_value_capped, dtype: float64

Upper fence: 397.9883039754379
Maximum after capping: 397.9883039754379


Q6 - Treat purchase_value outliers: Flag as feature + reflect
The analytics team wants to know if high-value orders tend to get better or worse reviews. Add an is_high_value_order flag and check if it correlates with review_score.

# Create df["is_high_value_order"] = True where purchase_value > upper_fence
# Print value_counts()
# Print df.groupby("is_high_value_order")["review_score"].mean()


In [18]:
df["is_high_value_order"] = df["purchase_value"] > upper_fence
print("Value counts:")
print(df["is_high_value_order"].value_counts())
df.groupby("is_high_value_order")["review_score"].mean()
print("\nAverage review score:")
print(df.groupby("is_high_value_order")["review_score"].mean())

print("\nFirst 5 high-value orders:")
print(df[df["is_high_value_order"]].head())

print("\nOriginal maximum purchase value:")
print(df["purchase_value"].max())

Value counts:
is_high_value_order
False    754
True      46
Name: count, dtype: int64

Average review score:
is_high_value_order
False    3.979045
True     3.967391
Name: review_score, dtype: float64

First 5 high-value orders:
     purchase_value  delivery_days  review_score  purchase_value_capped  \
8        560.395762       7.856667           4.2             397.988304   
16       437.757511       6.648644           4.3             397.988304   
47       414.072407       4.666291           4.0             397.988304   
62       449.842273       1.581624           4.1             397.988304   
104      644.891199       6.551485           3.4             397.988304   

     is_high_value_order  
8                   True  
16                  True  
47                  True  
62                  True  
104                 True  

Original maximum purchase value:
12500.0
